# 03 — Análisis de Patrones

## Objetivo
Descubrir patrones temporales, estacionales y correlaciones en los datos que nos ayuden a entender el comportamiento **normal** del sistema eléctrico ecuatoriano. Esto es crucial porque para detectar anomalías, primero debemos saber qué es "normal".

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])
df['mes'] = df['fecha'].dt.month
df['anio'] = df['fecha'].dt.year
df['trimestre'] = df['fecha'].dt.quarter
df['mes_nombre'] = df['fecha'].dt.strftime('%b')

## 3.1 Tendencia general

¿La demanda crece? ¿La composición del mix energético está cambiando?

In [ ]:
# Demanda anual promedio y tendencia
demanda_anual = df.groupby('anio')['demanda_twh'].agg(['mean', 'sum']).reset_index()
demanda_anual.columns = ['Año', 'Demanda media mensual (TWh)', 'Demanda anual (TWh)']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Demanda Mensual con Tendencia', 'Demanda Anual Acumulada'])

# Panel 1: serie mensual + rolling mean
fig.add_trace(go.Scatter(x=df['fecha'], y=df['demanda_twh'], name='Mensual',
                         line=dict(color='#90CAF9', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['fecha'], y=df['demanda_twh'].rolling(12).mean(), 
                         name='Media móvil 12m', line=dict(color='#1565C0', width=2.5)), row=1, col=1)

# Panel 2: barras anuales
fig.add_trace(go.Bar(x=demanda_anual['Año'], y=demanda_anual['Demanda anual (TWh)'],
                     name='Anual', marker_color='#1976D2'), row=1, col=2)

fig.update_layout(template='plotly_white', height=400, title_text='Tendencia de Demanda Eléctrica')
fig.show()

# Tasa de crecimiento
print("Crecimiento interanual de demanda:")
for i in range(1, len(demanda_anual)):
    prev = demanda_anual.iloc[i-1]['Demanda anual (TWh)']
    curr = demanda_anual.iloc[i]['Demanda anual (TWh)']
    growth = (curr - prev) / prev * 100
    print(f"  {int(demanda_anual.iloc[i]['Año'])}: {growth:+.1f}%")

## 3.2 Estacionalidad

Ecuador tiene dos estaciones climáticas principales:
- **Lluviosa** (oct-may): Más agua → más generación hidro
- **Seca** (jun-sep): Menos agua → más generación térmica

¿Se refleja esto en los datos?

In [ ]:
# Estacionalidad por mes
mes_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'Demanda por Mes', 'Generación Hidro por Mes (%)', 'Generación Fósil por Mes (%)'])

for i, (col, color) in enumerate([
    ('demanda_twh', '#1976D2'), ('gen_hydro', '#2196F3'), ('gen_fossil', '#F44336')
], 1):
    monthly_avg = df.groupby('mes')[col].mean().reset_index()
    fig.add_trace(go.Bar(x=monthly_avg['mes'], y=monthly_avg[col],
                         marker_color=color, name=col), row=1, col=i)

fig.update_layout(template='plotly_white', height=400, showlegend=False,
                  title_text='Patrones Estacionales del Sector Eléctrico Ecuatoriano')
fig.show()

## 3.3 Correlaciones

¿Qué variables se mueven juntas? Esto nos dice qué features serán redundantes y cuáles aportan información independiente.

In [ ]:
# Matriz de correlación
cols_corr = ['demanda_twh', 'gen_hydro', 'gen_fossil', 'gen_gas', 'gen_other_fossil',
             'gen_wind', 'gen_bioenergy', 'gen_total_generation',
             'co2_intensity_gco2_kwh', 'importaciones_netas_twh']
cols_corr = [c for c in cols_corr if c in df.columns]

corr = df[cols_corr].corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                title='Matriz de Correlación — Variables del Sector Eléctrico',
                labels={'color': 'Correlación'}, zmin=-1, zmax=1)
fig.update_layout(template='plotly_white', height=600, width=700)
fig.show()

In [ ]:
# Scatter: Hidro vs Fósil — la relación inversa clave
fig = px.scatter(df, x='gen_hydro', y='gen_fossil', color='anio',
                 size='demanda_twh', hover_data=['fecha'],
                 title='Hidro vs Fósil — Relación Inversa (tamaño = demanda)',
                 labels={'gen_hydro': 'Generación Hidro (%)', 'gen_fossil': 'Generación Fósil (%)'})
fig.update_layout(template='plotly_white')
fig.show()

print(f"Correlación hidro-fósil: {df['gen_hydro'].corr(df['gen_fossil']):.3f}")
print("→ Correlación fuertemente negativa: cuando baja hidro, sube fósil (compensación)")

## 3.4 Descomposición de serie temporal

Separamos la demanda en: **tendencia** + **estacionalidad** + **residual**. Los residuales grandes son candidatos a anomalías.

In [ ]:
# Descomposición manual (media móvil como trend)
df_sorted = df.sort_values('fecha').copy()
df_sorted['trend'] = df_sorted['demanda_twh'].rolling(12, center=True, min_periods=6).mean()
df_sorted['residual'] = df_sorted['demanda_twh'] - df_sorted['trend']

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['Demanda Original', 'Tendencia (media móvil 12m)', 'Residual'],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df_sorted['fecha'], y=df_sorted['demanda_twh'],
                         line=dict(color='#1976D2')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_sorted['fecha'], y=df_sorted['trend'],
                         line=dict(color='#4CAF50', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=df_sorted['fecha'], y=df_sorted['residual'],
                         line=dict(color='#FF9800')), row=3, col=1)

# Marcar residuales extremos (>2 std)
std_res = df_sorted['residual'].std()
extremos = df_sorted[df_sorted['residual'].abs() > 2 * std_res]
fig.add_trace(go.Scatter(x=extremos['fecha'], y=extremos['residual'], mode='markers',
                         marker=dict(color='red', size=10, symbol='x'),
                         name='Residual extremo (>2σ)'), row=3, col=1)

fig.update_layout(template='plotly_white', height=600, showlegend=False,
                  title_text='Descomposición de la Demanda Eléctrica')
fig.show()

print(f"Desviación estándar del residual: {std_res:.3f} TWh")
print(f"Meses con residual extremo (>2σ): {len(extremos)}")
if len(extremos) > 0:
    for _, row in extremos.iterrows():
        print(f"  {row['fecha'].strftime('%Y-%m')}: residual = {row['residual']:.3f} TWh")

## 3.5 Hallazgos clave

1. **Tendencia creciente** — La demanda crece ~3-5% anual, con excepción de 2020 (COVID) y 2024 (crisis)
2. **Estacionalidad moderada** — Meses de mitad de año muestran mayor participación fósil (estación seca de la Sierra)
3. **Relación hidro-fósil inversa** — Correlación ~-0.99: son un espejo perfecto. Cuando no hay agua, se encienden las térmicas
4. **CO₂ como indicador** — La intensidad de carbono está directamente ligada al mix fósil, útil como feature
5. **Residuales extremos** — Los picos en residuales coinciden con eventos reales (crisis, COVID)

**Para el modelo:** Necesitamos features que capturen la estacionalidad, la tendencia, y las desviaciones del patrón normal. Isolation Forest puede trabajar con múltiples dimensiones simultáneamente.

---
*Siguiente notebook: 04 — Feature Engineering*